# Demand & Inventory Intelligence System
## Retail Demand Forecasting and Inventory Risk Prediction

---

### Phase 4 — Data Integration & Common Analytical Model (CAM)

**Project:** FORESIGHT · **Phase:** 4 · **Input:** Phase 3 cleaned datasets (`data/processed/`)

The Common Analytical Model (CAM) is the standardized, source-aware star-schema layer
between the cleaned data and every downstream consumer — EDA, feature engineering, ML
forecasting, inventory risk, Power BI, and Streamlit.

**Guiding rules applied in this phase:**
- Only Phase 3 processed datasets are used; raw files are never read.
- UCI and Synthetic data are **never blindly concatenated** — every row carries a
  mandatory `source_dataset` discriminator (`UCI` / `SYNTHETIC`).
- No fake values are fabricated: UCI gets no fake stores, categories, suppliers, lead
  times, reorder points, safety stock, promotions, or inventory.
- The Phase 3 **inventory REVIEW semantic** is preserved (`ending_inventory =
  beginning_inventory − units_sold`; receipts are never re-added).
- Guest transactions, returns, and cancellations are preserved and kept separate.
- No lag / rolling ML features are created yet — those belong to Phase 6.

### 2. Integration Objectives

1. **One contract, many consumers** — a single standardized layer feeding EDA, ML,
   risk, Power BI, and Streamlit without duplicated preprocessing.
2. **Source identity** — every fact/dimension row is attributable to `UCI` or `SYNTHETIC`.
3. **Source-aware keys** — `UCI_<StockCode>`, `SYN_<sku_id>`, `SYN_<customer_id>`, and the
   single `ONLINE` channel entity (no fake stores).
4. **Honest NULLs** — fields that do not exist for a source stay NULL, never invented.
5. **Validated relationships** — grain, primary keys, foreign keys, and business rules
   all verified with 0 unexplained orphans.
6. **Reproducibility** — the pipeline is a reusable module
   (`src/data_integration.py`) plus this executable notebook.

### 3. Load Clean Data

Load every Phase 3 processed dataset through the cached loaders in `src/data_integration.py` (parquet for the large fact files, CSV for the small dimension files).

In [1]:
# ---- Environment & project root ------------------------------
import os, sys, json, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
# Robust project-root detection (works whether launched from notebooks/ or root).
_BASE = None
for _p in (os.getcwd(), os.path.dirname(os.getcwd()), os.path.dirname(os.path.dirname(os.getcwd()))):
    if os.path.isfile(os.path.join(_p, "src", "data_integration.py")):
        _BASE = os.path.abspath(_p)
        break
if _BASE is None:
    raise RuntimeError("Could not locate project root (src/data_integration.py).")
if _BASE not in sys.path:
    sys.path.insert(0, _BASE)

from src import data_integration as di
from src.cam_adapter import compatibility_report

In [2]:
# ---- Load all Phase 3 clean datasets --------------------------
data = di.load_clean_data()
summary = pd.DataFrame([
    {"Dataset": "store_master",       "Rows": len(data["store_master"]),       "Cols": len(data["store_master"].columns)},
    {"Dataset": "sku_master",         "Rows": len(data["sku_master"]),         "Cols": len(data["sku_master"].columns)},
    {"Dataset": "customer_master",    "Rows": len(data["customer_master"]),    "Cols": len(data["customer_master"].columns)},
    {"Dataset": "calendar",           "Rows": len(data["calendar"]),           "Cols": len(data["calendar"].columns)},
    {"Dataset": "sales_daily",        "Rows": len(data["sales_daily"]),        "Cols": len(data["sales_daily"].columns)},
    {"Dataset": "inventory_snapshots","Rows": len(data["inventory_snapshots"]),"Cols": len(data["inventory_snapshots"].columns)},
    {"Dataset": "uci_sales",          "Rows": len(data["uci_sales"]),          "Cols": len(data["uci_sales"].columns)},
    {"Dataset": "uci_returns",        "Rows": len(data["uci_returns"]),        "Cols": len(data["uci_returns"].columns)},
    {"Dataset": "uci_cancellations",  "Rows": len(data["uci_cancellations"]),  "Cols": len(data["uci_cancellations"].columns)},
])
summary

,Dataset,Rows,Cols
0,store_master,30,8
1,sku_master,5000,12
2,customer_master,10000,5
3,calendar,1461,13
4,sales_daily,1461000,9
5,inventory_snapshots,1461000,11
6,uci_sales,1010533,18
7,uci_returns,3393,18
8,uci_cancellations,19104,18


### 4. Review Phase 3 Quality Results

Phase 3 recorded per-dataset quality status. Key flags carried into the CAM:

In [3]:
# ---- Phase 3 quality status summary ----------------------------
_qr = os.path.join(_BASE, "docs", "data_quality_report.json")
if os.path.exists(_qr):
    q = json.load(open(_qr, encoding="utf-8"))
    rows = []
    for name, r in q.items():
        if isinstance(r, dict):
            rows.append({"dataset": name,
                         "quality_status": r.get("quality_status", "n/a"),
                         "final_rows": r.get("final_rows", r.get("rows", "n/a"))})
    qdf = pd.DataFrame(rows)
    print("Phase 3 per-dataset quality status:")
    print(qdf.to_string(index=False))
else:
    print("data_quality_report.json not found.")
print()
print("Phase 3 UCI transaction split:",
      sorted(data["uci_sales"]["transaction_type"].unique()),
      "| returns:", sorted(data["uci_returns"]["transaction_type"].unique()),
      "| cancellations:", sorted(data["uci_cancellations"]["transaction_type"].unique()))
print("Phase 3 inventory REVIEW semantic: beginning_inventory already includes the day's",
      "receipts; ending_inventory = beginning_inventory - units_sold.")

Phase 3 per-dataset quality status:
            dataset quality_status  final_rows
   online_retail_ii         REVIEW     1033036
       store_master           PASS          30
         sku_master           PASS        5000
    customer_master           PASS       10000
           calendar           PASS        1461
        sales_daily           PASS     1461000
inventory_snapshots         REVIEW     1461000

Phase 3 UCI transaction split: ['SALE'] | returns: ['RETURN'] | cancellations: ['CANCELLATION']
Phase 3 inventory REVIEW semantic: beginning_inventory already includes the day's receipts; ending_inventory = beginning_inventory - units_sold.


### 5. Define Common Analytical Model

The CAM is a **star schema**: shared dimensions (`dim_calendar`, `dim_product`, `dim_entity`, `dim_customer`) feeding analytical facts (`fact_sales`, `fact_inventory`, `fact_returns`, `fact_cancellations`) and derived analytical tables (`inventory_analytics`, `customer_analytics`, `forecast_base`). Every table preserves `source_dataset`.

In [4]:
# ---- CAM table map ---------------------------------------------
CAM_TABLES = {
    "dim_calendar":         "Calendar dimension (synthetic + UCI date range)",
    "dim_product":          "Source-aware product dimension (SYN_/UCI_ keys)",
    "dim_entity":           "Entity dimension (30 stores + ONLINE channel)",
    "dim_customer":         "Customer dimension (identified customers only)",
    "fact_sales":           "Daily demand fact (UCI online + synthetic stores)",
    "fact_inventory":       "Daily inventory fact (Synthetic only)",
    "fact_returns":         "UCI returns fact (separate from demand)",
    "fact_cancellations":   "UCI cancellations fact (separate from demand)",
    "inventory_analytics":  "Inventory joined to product attrs (risk prep)",
    "customer_analytics":   "Identified-customer metrics",
    "forecast_base":        "Standardized forecasting input (no lag features)",
}
for t, desc in CAM_TABLES.items():
    print(f"{t:<22} {desc}")

dim_calendar           Calendar dimension (synthetic + UCI date range)
dim_product            Source-aware product dimension (SYN_/UCI_ keys)
dim_entity             Entity dimension (30 stores + ONLINE channel)
dim_customer           Customer dimension (identified customers only)
fact_sales             Daily demand fact (UCI online + synthetic stores)
fact_inventory         Daily inventory fact (Synthetic only)
fact_returns           UCI returns fact (separate from demand)
fact_cancellations     UCI cancellations fact (separate from demand)
inventory_analytics    Inventory joined to product attrs (risk prep)
customer_analytics     Identified-customer metrics
forecast_base          Standardized forecasting input (no lag features)


### 6. Define Analytical Grain

The primary analytical grain for the fact tables is **`date + source_dataset + entity_id + product_key`**. Dimension primary keys are listed here and validated later.

In [5]:
# ---- Grain / primary key definitions ---------------------------
grains = {
    "dim_calendar":         ["date"],
    "dim_product":          ["product_key"],
    "dim_entity":           ["source_dataset", "entity_id"],
    "dim_customer":         ["customer_key"],
    "fact_sales":           ["date", "source_dataset", "entity_id", "product_key"],
    "fact_inventory":       ["date", "source_dataset", "entity_id", "product_key"],
    "fact_returns":         ["date", "source_dataset", "entity_id", "product_key"],
    "fact_cancellations":   ["date", "source_dataset", "entity_id", "product_key"],
    "inventory_analytics":  ["date", "source_dataset", "entity_id", "product_key"],
    "customer_analytics":   ["customer_key"],
    "forecast_base":        ["date", "source_dataset", "entity_id", "product_key"],
}
pd.DataFrame([{"table": k, "primary_key / grain": " + ".join(v)} for k, v in grains.items()])

,table,primary_key / grain
0,dim_calendar,date
1,dim_product,product_key
2,dim_entity,source_dataset + entity_id
3,dim_customer,customer_key
4,fact_sales,date + source_dataset + entity_id + product_key
5,fact_inventory,date + source_dataset + entity_id + product_key
6,fact_returns,date + source_dataset + entity_id + product_key
7,fact_cancellations,date + source_dataset + entity_id + product_key
8,inventory_analytics,date + source_dataset + entity_id + product_key
9,customer_analytics,customer_key


### 7. Create Dimension Tables

* `dim_calendar` — the Phase 3 cleaned synthetic calendar extended over the UCI
  transaction date range with pure calendar-math attributes (no fabricated holidays).
* `dim_product` — source-aware keys `SYN_<sku_id>` / `UCI_<StockCode>`; UCI rows only
  populate available fields (category/brand/supplier/lead time/reorder/safety = NULL).
* `dim_entity` — 30 synthetic stores + the single `ONLINE` channel (no fake stores).
* `dim_customer` — identified customers only; guests stay out of the dimension and are
  flagged in facts.

In [6]:
# ---- Build the four dimensions --------------------------------
tables = {
    "dim_calendar": di.create_calendar_dimension(),
    "dim_product":  di.create_product_dimension(),
    "dim_entity":   di.create_entity_dimension(),
    "dim_customer": di.create_customer_dimension(),
}
for t, df in tables.items():
    srcs = sorted(df["source_dataset"].astype(str).unique()) if "source_dataset" in df else "n/a"
    print(f"{t:<14} rows={len(df):>8,}  src={srcs}")
print()
print("dim_calendar span:", tables["dim_calendar"]["date"].min().date(), "->",
      tables["dim_calendar"]["date"].max().date(),
      "| UCI_DERIVED rows:", int((tables["dim_calendar"]["date_source"] == "UCI_DERIVED").sum()))
uci_ent = tables["dim_entity"][tables["dim_entity"]["source_dataset"] == "UCI"].to_dict("records")[0]
print("dim_entity UCI row:", {k: uci_ent[k] for k in ("entity_id", "entity_type", "source_dataset", "store_name")})
print("dim_product UCI sample:",
      tables["dim_product"][tables["dim_product"]["source_dataset"] == "UCI"][["product_key", "product_name"]].head(3).to_dict("records"))

dim_calendar   rows=   2,200  src=n/a
dim_product    rows=  10,304  src=['SYNTHETIC', 'UCI']
dim_entity     rows=      31  src=['SYNTHETIC', 'UCI']
dim_customer   rows=  15,881  src=['SYNTHETIC', 'UCI']

dim_calendar span: 2009-12-01 -> 2025-12-31 | UCI_DERIVED rows: 739
dim_entity UCI row: {'entity_id': 'ONLINE', 'entity_type': 'CHANNEL', 'source_dataset': 'UCI', 'store_name': None}
dim_product UCI sample: [{'product_key': 'UCI_85123A', 'product_name': 'WHITE HANGING HEART T-LIGHT HOLDER'}, {'product_key': 'UCI_22423', 'product_name': 'REGENCY CAKESTAND 3 TIER'}, {'product_key': 'UCI_85099B', 'product_name': 'JUMBO BAG RED RETROSPOT'}]


### 8. Create Sales Fact

`fact_sales` aggregates UCI transactions to DATE + SKU (`entity_id = ONLINE`) and maps synthetic daily store-SKU sales to `SYN_<sku_id>` product keys. Returns/cancellations are never included; guest UCI transactions stay in sales but are not counted as identified customers.

In [7]:
# ---- Build the sales fact --------------------------------------
tables["fact_sales"] = di.create_sales_fact()
sf = tables["fact_sales"]
print("fact_sales rows:", f"{len(sf):,}",
      "| UCI:", f"{int((sf.source_dataset=='UCI').sum()):,}",
      "| SYNTHETIC:", f"{int((sf.source_dataset=='SYNTHETIC').sum()):,}")
print("UCI promotion_flag all NULL (no promotion source):", bool(sf.loc[sf.source_dataset=='UCI', 'promotion_flag'].isna().all()))
sf[sf.source_dataset == "UCI"].head(3)

fact_sales rows: 1,995,496 | UCI: 534,496 | SYNTHETIC: 1,461,000
UCI promotion_flag all NULL (no promotion source): True


,date,source_dataset,entity_id,entity_type,product_key,sku_id,units_sold,revenue,average_unit_price,transaction_count,unique_customers,promotion_flag
0,2009-12-01,UCI,ONLINE,CHANNEL,UCI_10002,10002,12,10.2,0.85,1,1,<NA>
1,2009-12-01,UCI,ONLINE,CHANNEL,UCI_10120,10120,60,12.6,0.21,1,1,<NA>
2,2009-12-01,UCI,ONLINE,CHANNEL,UCI_10123C,10123C,3,3.9,1.30,1,0,<NA>


In [8]:
sf[sf.source_dataset == "SYNTHETIC"].head(3)

,date,source_dataset,entity_id,entity_type,product_key,sku_id,units_sold,revenue,average_unit_price,transaction_count,unique_customers,promotion_flag
534496,2022-01-01,SYNTHETIC,STORE_001,STORE,SYN_SKU_00001,SKU_00001,26,3717.74,142.99,15,13,0
534497,2022-01-02,SYNTHETIC,STORE_001,STORE,SYN_SKU_00001,SKU_00001,14,2001.86,142.99,8,7,0
534498,2022-01-03,SYNTHETIC,STORE_001,STORE,SYN_SKU_00001,SKU_00001,7,1000.93,142.99,4,3,0


### 9. Create Inventory Fact

`fact_inventory` is populated **only** by the Synthetic source (UCI has no native inventory — no fake records). It preserves the Phase 3 REVIEW semantic and its derived columns.

In [9]:
# ---- Build the inventory fact ----------------------------------
tables["fact_inventory"] = di.create_inventory_fact()
inv = tables["fact_inventory"]
print("fact_inventory rows:", f"{len(inv):,}", "| sources:", sorted(inv.source_dataset.unique()))
print("inventory_balance_ok rate: {:.4f}".format(inv.inventory_balance_ok.mean()))
_bal = inv[inv.inventory_balance_ok]
print("REVIEW semantic: ending == beginning - units_sold on balanced rows ->",
      bool((_bal["ending_inventory"] == _bal["beginning_inventory"] - _bal["units_sold"]).all()))
inv.head(3)

fact_inventory rows: 1,461,000 | sources: ['SYNTHETIC']
inventory_balance_ok rate: 1.0000
REVIEW semantic: ending == beginning - units_sold on balanced rows -> True


,date,source_dataset,entity_id,product_key,sku_id,beginning_inventory,beginning_inventory_pre_receipts,receipts,units_sold,ending_inventory,stockout_flag,on_order_qty,inventory_balance_ok
0,2022-01-01,SYNTHETIC,STORE_001,SYN_SKU_00001,SKU_00001,84,84,0,26,58,0,0,True
1,2022-01-02,SYNTHETIC,STORE_001,SYN_SKU_00001,SKU_00001,58,58,0,14,44,0,100,True
2,2022-01-03,SYNTHETIC,STORE_001,SYN_SKU_00001,SKU_00001,44,44,0,7,37,0,100,True


**Inventory analytical table.** `inventory_analytics` joins the inventory fact to product attributes (category/sub-category/brand, lead time, reorder point, safety stock). It *prepares* data for Phase 10 risk scoring — it does **not** compute risk scores here.

In [10]:
# ---- Inventory analytical table (risk prep) ---------------------
tables["inventory_analytics"] = di.create_inventory_analytics(
    tables["fact_inventory"], tables["dim_product"])
ia = tables["inventory_analytics"]
print("inventory_analytics rows:", f"{len(ia):,}", "| columns:", list(ia.columns))
print("no risk-score columns (Phase 10):",
      not any("risk" in c or "score" in c for c in ia.columns))
ia.head(3)

inventory_analytics rows: 1,461,000 | columns: ['date', 'source_dataset', 'entity_id', 'product_key', 'sku_id', 'category', 'sub_category', 'brand', 'ending_inventory', 'on_order_qty', 'stockout_flag', 'lead_time_days', 'reorder_point', 'safety_stock']
no risk-score columns (Phase 10): True


,date,source_dataset,entity_id,product_key,sku_id,category,sub_category,brand,ending_inventory,on_order_qty,stockout_flag,lead_time_days,reorder_point,safety_stock
0,2022-01-01,SYNTHETIC,STORE_001,SYN_SKU_00001,SKU_00001,Apparel,Accessories,Brand_CE,58,0,0,20,56,8
1,2022-01-02,SYNTHETIC,STORE_001,SYN_SKU_00001,SKU_00001,Apparel,Accessories,Brand_CE,44,100,0,20,56,8
2,2022-01-03,SYNTHETIC,STORE_001,SYN_SKU_00001,SKU_00001,Apparel,Accessories,Brand_CE,37,100,0,20,56,8


### 10. Create Customer Analytics

`customer_analytics` holds identified-customer metrics. UCI metrics are computed where a Customer ID exists (guest transactions excluded). The Synthetic source has no customer-grain transactions in the processed data, so its transaction metrics are NULL (never fabricated).

In [11]:
# ---- Build customer analytics ---------------------------------
tables["customer_analytics"] = di.create_customer_analytics()
ca = tables["customer_analytics"]
print("customer_analytics rows:", f"{len(ca):,}",
      "| UCI:", int((ca.source_dataset=='UCI').sum()),
      "| SYNTHETIC:", int((ca.source_dataset=='SYNTHETIC').sum()))
print("null customer_key rows:", int(ca.customer_key.isna().sum()))
ca[ca.source_dataset == "UCI"].head(3)

customer_analytics rows: 15,881 | UCI: 5881 | SYNTHETIC: 10000
null customer_key rows: 0


,customer_key,source_dataset,customer_id,customer_segment,loyalty_member,signup_date,country,transaction_count,total_units,total_revenue,first_purchase_date,last_purchase_date
0,UCI_12346,UCI,12346,<NA>,<NA>,<NA>,United Kingdom,12,74285,77556.46,2009-12-14,2011-01-18
1,UCI_12347,UCI,12347,<NA>,<NA>,<NA>,Iceland,8,2967,4921.53,2010-10-31,2011-12-07
2,UCI_12348,UCI,12348,<NA>,<NA>,<NA>,Finland,5,2714,2019.4,2010-09-27,2011-09-25


### 11. Create Returns Fact

`fact_returns` comes from the UCI returns split, aggregated to DATE + SKU, and is kept **strictly separate** from demand sales. (All Phase 3 UCI returns are guest, UK, price-zero lines — revenue impact is therefore 0.0, documented, not dropped.)

In [12]:
# ---- Build the returns fact ------------------------------------
tables["fact_returns"] = di.create_returns_fact()
rt = tables["fact_returns"]
print("fact_returns rows:", f"{len(rt):,}",
      "| quantity_returned:", int(rt.quantity_returned.sum()),
      "| revenue_impact:", float(rt.revenue_impact.sum()),
      "| source:", list(rt.source_dataset.unique()))
rt.head(3)

fact_returns rows: 3,338 | quantity_returned: 569314 | revenue_impact: 0.0 | source: ['UCI']


,date,source_dataset,entity_id,entity_type,product_key,sku_id,quantity_returned,return_transactions,revenue_impact
0,2009-12-01,UCI,ONLINE,CHANNEL,UCI_20683,20683,44,1,0.0
1,2009-12-01,UCI,ONLINE,CHANNEL,UCI_21646,21646,50,1,0.0
2,2009-12-01,UCI,ONLINE,CHANNEL,UCI_21733,21733,96,1,0.0


### 12. Create Cancellations Fact

`fact_cancellations` comes from the UCI cancellations split and is kept separate from demand. The documented **anomalous cancellation line** (Invoice `C496350`, StockCode `M`, quantity `+1`, price `373.57`) is preserved — it appears as `UCI_M` on 2010-02-01 and is never silently removed.

In [13]:
# ---- Build the cancellations fact ------------------------------
tables["fact_cancellations"] = di.create_cancellations_fact()
cc = tables["fact_cancellations"]
print("fact_cancellations rows:", f"{len(cc):,}",
      "| cancelled_quantity:", int(cc.cancelled_quantity.sum()),
      "| revenue_impact:", float(cc.revenue_impact.sum()))
anom = cc[(cc.date == pd.Timestamp("2010-02-01")) & (cc.product_key == "UCI_M")]
print("Anomalous cancellation group (UCI_M / 2010-02-01) present:", len(anom) == 1)
cc.head(3)

fact_cancellations rows: 17,132 | cancelled_quantity: 476821 | revenue_impact: -1462050.6099999999
Anomalous cancellation group (UCI_M / 2010-02-01) present: True


,date,source_dataset,entity_id,entity_type,product_key,sku_id,cancelled_quantity,cancellation_transactions,revenue_impact
0,2009-12-01,UCI,ONLINE,CHANNEL,UCI_15056N,15056N,1,1,-5.95
1,2009-12-01,UCI,ONLINE,CHANNEL,UCI_20682,20682,1,1,-3.25
2,2009-12-01,UCI,ONLINE,CHANNEL,UCI_20686,20686,1,1,-3.25


### 13. Create Forecast Base Dataset

`forecast_base` is the standardized input for future forecasting — a projection of `fact_sales` with the exact downstream column contract. It deliberately contains **no lag / rolling / EWM features** (Phase 6).

In [14]:
# ---- Build the forecast base -----------------------------------
tables["forecast_base"] = di.create_forecast_base(tables["fact_sales"])
fb = tables["forecast_base"]
print("forecast_base rows:", f"{len(fb):,}")
print("columns:", list(fb.columns))
print("leaky feature cols present:", [c for c in fb.columns if "lag_" in c or "rolling_" in c or "_ewm_" in c])
fb.head(3)

forecast_base rows: 1,995,496
columns: ['date', 'source_dataset', 'entity_id', 'entity_type', 'product_key', 'sku_id', 'units_sold', 'revenue', 'average_unit_price', 'transaction_count', 'unique_customers', 'promotion_flag']
leaky feature cols present: []


,date,source_dataset,entity_id,entity_type,product_key,sku_id,units_sold,revenue,average_unit_price,transaction_count,unique_customers,promotion_flag
0,2009-12-01,UCI,ONLINE,CHANNEL,UCI_10002,10002,12,10.2,0.85,1,1,<NA>
1,2009-12-01,UCI,ONLINE,CHANNEL,UCI_10120,10120,60,12.6,0.21,1,1,<NA>
2,2009-12-01,UCI,ONLINE,CHANNEL,UCI_10123C,10123C,3,3.9,1.30,1,0,<NA>


### 14. Validate Relationships

Foreign-key validation across the star schema — target **0 unexplained orphan records**.

In [15]:
# ---- Foreign-key validation ------------------------------------
fks = [
    ("fact_sales",          "dim_calendar", ["date"], ["date"]),
    ("fact_sales",          "dim_product",  ["product_key"], ["product_key"]),
    ("fact_sales",          "dim_entity",   ["source_dataset", "entity_id"], ["source_dataset", "entity_id"]),
    ("fact_inventory",      "dim_calendar", ["date"], ["date"]),
    ("fact_inventory",      "dim_product",  ["product_key"], ["product_key"]),
    ("fact_inventory",      "dim_entity",   ["source_dataset", "entity_id"], ["source_dataset", "entity_id"]),
    ("fact_returns",        "dim_calendar", ["date"], ["date"]),
    ("fact_returns",        "dim_product",  ["product_key"], ["product_key"]),
    ("fact_cancellations",  "dim_calendar", ["date"], ["date"]),
    ("fact_cancellations",  "dim_product",  ["product_key"], ["product_key"]),
]
fk_rows = []
for fact, dim, fk, dk in fks:
    res = di.validate_foreign_keys(tables[fact], tables[dim], fk, dk, fact, dim)
    fk_rows.append({"fact": fact, "dimension": dim, "key": " + ".join(fk), "orphan_count": res["orphan_count"]})
fkdf = pd.DataFrame(fk_rows)
fkdf

,fact,dimension,key,orphan_count
0,fact_sales,dim_calendar,date,0
1,fact_sales,dim_product,product_key,0
2,fact_sales,dim_entity,source_dataset + entity_id,0
3,fact_inventory,dim_calendar,date,0
4,fact_inventory,dim_product,product_key,0
5,fact_inventory,dim_entity,source_dataset + entity_id,0
6,fact_returns,dim_calendar,date,0
7,fact_returns,dim_product,product_key,0
8,fact_cancellations,dim_calendar,date,0
9,fact_cancellations,dim_product,product_key,0


In [16]:
print("Total orphans across all relationships:", int(fkdf.orphan_count.sum()))

Total orphans across all relationships: 0


### 15. Validate Grain

Report row count, duplicate keys, and null keys per table.

In [17]:
# ---- Grain validation -------------------------------------------
grain_rows = []
for name, key in grains.items():
    grain_rows.append(di.validate_grain(tables[name], key, name))
gdf = pd.DataFrame(grain_rows)
gdf

,table,grain,row_count,duplicate_key_count,null_key_count
0,dim_calendar,date,2200,0,0
1,dim_product,product_key,10304,0,0
2,dim_entity,source_dataset+entity_id,31,0,0
3,dim_customer,customer_key,15881,0,0
4,fact_sales,date+source_dataset+entity_id+product_key,1995496,0,0
5,fact_inventory,date+source_dataset+entity_id+product_key,1461000,0,0
6,fact_returns,date+source_dataset+entity_id+product_key,3338,0,0
7,fact_cancellations,date+source_dataset+entity_id+product_key,17132,0,0
8,inventory_analytics,date+source_dataset+entity_id+product_key,1461000,0,0
9,customer_analytics,customer_key,15881,0,0


### 16. Validate Business Rules

Business rules capture the Phase 3 REVIEW inventory semantic, source separation, guest handling, the anomalous cancellation line, and the no-lag contract on `forecast_base`.

In [18]:
# ---- Business-rule validation ----------------------------------
rules = di.validate_business_rules(tables)
rdf = pd.DataFrame([{"rule": r[0], "passed": bool(r[1]), "detail": r[2]} for r in rules])
rdf

,rule,passed,detail
0,inventory_equation_ending_eq_beginning_minus_u...,True,"1,461,000 balanced rows verified"
1,receipts_not_rea_added_to_ending,True,"122,208 receipt-days checked"
2,sales_units_nonnegative,True,0 negatives
3,source_dataset_enum_valid,True,"['SYNTHETIC', 'UCI']"
4,uci_promotion_flag_null_not_invented,True,UCI has no promotion source -> NULL
5,returns_separated_from_sales,True,"3,338 return rows"
6,cancellations_separated_from_sales,True,"17,132 cancellation rows"
7,cancellation_anomaly_preserved,True,source line C496350 preserved: True; fact grou...
8,inventory_synthetic_only_no_fake_uci_inventory,True,['SYNTHETIC']
9,customer_analytics_no_null_customer_key,True,0 null keys


In [19]:
print("Business rules passed:", int(rdf.passed.sum()), "/", len(rdf))

Business rules passed: 11 / 11


### 17. Save Integrated Data

Persist every CAM table as Parquet under `data/processed/integrated/`.

In [20]:
# ---- Persist CAM tables to parquet ------------------------------
written = di.save_integrated_data(tables)
pd.DataFrame([{"table": k, "path": v, "size_bytes": os.path.getsize(v)} for k, v in written.items()])

,table,path,size_bytes
0,dim_calendar,C:\Users\SURAG\Documents\zidio\Project_FORESIG...,28627
1,dim_product,C:\Users\SURAG\Documents\zidio\Project_FORESIG...,385751
2,dim_entity,C:\Users\SURAG\Documents\zidio\Project_FORESIG...,7660
3,dim_customer,C:\Users\SURAG\Documents\zidio\Project_FORESIG...,260204
4,fact_sales,C:\Users\SURAG\Documents\zidio\Project_FORESIG...,11120443
5,fact_inventory,C:\Users\SURAG\Documents\zidio\Project_FORESIG...,5855179
6,inventory_analytics,C:\Users\SURAG\Documents\zidio\Project_FORESIG...,3247326
7,customer_analytics,C:\Users\SURAG\Documents\zidio\Project_FORESIG...,336401
8,fact_returns,C:\Users\SURAG\Documents\zidio\Project_FORESIG...,52493
9,fact_cancellations,C:\Users\SURAG\Documents\zidio\Project_FORESIG...,143387


### 18. Integration Quality Report

Generate `docs/integration_quality_report.json` and `docs/integration_quality_report.csv` including the `inventory_data_status = REVIEW` metadata.

In [21]:
# ---- Integration quality report ---------------------------------
report = di.generate_integration_report(tables)
qdf = pd.DataFrame(report["tables"])
print("inventory_data_status:", report["inventory_data_status"])
qdf

inventory_data_status: REVIEW


,table_name,source_dataset,row_count,column_count,primary_key,duplicate_keys,null_keys,foreign_key_violations,grain,status
0,dim_calendar,N/A,2200,13,date,0,0,0,date,PASS
1,dim_product,"SYNTHETIC,UCI",10304,14,product_key,0,0,0,product_key,PASS
2,dim_entity,"SYNTHETIC,UCI",31,10,source_dataset+entity_id,0,0,0,source_dataset+entity_id,PASS
3,dim_customer,"SYNTHETIC,UCI",15881,9,customer_key,0,0,0,customer_key,PASS
4,fact_sales,"SYNTHETIC,UCI",1995496,12,date+source_dataset+entity_id+product_key,0,0,0,date+source_dataset+entity_id+product_key,PASS
5,fact_inventory,SYNTHETIC,1461000,13,date+source_dataset+entity_id+product_key,0,0,0,date+source_dataset+entity_id+product_key,PASS
6,inventory_analytics,SYNTHETIC,1461000,14,date+source_dataset+entity_id+product_key,0,0,0,date+source_dataset+entity_id+product_key,PASS
7,customer_analytics,"SYNTHETIC,UCI",15881,12,customer_key,0,0,0,customer_key,PASS
8,fact_returns,UCI,3338,9,date+source_dataset+entity_id+product_key,0,0,0,date+source_dataset+entity_id+product_key,PASS
9,fact_cancellations,UCI,17132,9,date+source_dataset+entity_id+product_key,0,0,0,date+source_dataset+entity_id+product_key,PASS


In [22]:
print("Report files:", report["_files"])
print("Business rules:", report["rule_passed"], "/", report["rule_total"])
print()
rep = compatibility_report()
print("ML forecasting compatible:", rep["ml_forecasting"]["compatible"])
print("Inventory risk compatible:", rep["inventory_risk"]["compatible"])
print("Streamlit contract documented:", bool(rep["streamlit"]["required_data_files"]))

Report files: {'json': 'C:\\Users\\SURAG\\Documents\\zidio\\Project_FORESIGHT\\Demand-Inventory-Intelligence\\docs\\integration_quality_report.json', 'csv': 'C:\\Users\\SURAG\\Documents\\zidio\\Project_FORESIGHT\\Demand-Inventory-Intelligence\\docs\\integration_quality_report.csv'}
Business rules: 11 / 11



ML forecasting compatible: True
Inventory risk compatible: True
Streamlit contract documented: True


### 19. Final Summary

**Phase 4 — Data Integration & Common Analytical Model — complete.**

- **11 tables** created under `data/processed/integrated/` (4 dimensions + 4 facts + 3 analytical tables).
- **Source identity preserved** — every fact/dimension row carries `source_dataset`
  (`UCI` / `SYNTHETIC`); keys are collision-safe (`UCI_<StockCode>`, `SYN_<sku_id>`,
  `SYN_<customer_id>`, single `ONLINE` channel).
- **No fabricated values** — UCI categories/brands/suppliers/lead times/reorder
  points/safety stock/promotions/inventory all remain NULL where unavailable.
- **Phase 3 REVIEW semantic preserved** — `ending_inventory = beginning_inventory −
  units_sold`; receipts never re-added; `beginning_inventory_pre_receipts` and
  `inventory_balance_ok` carried through.
- **Returns & cancellations kept separate**; the anomalous cancellation line
  (`C496350`) is preserved.
- **0 duplicate keys, 0 null keys, 0 foreign-key orphans** across all tables; all
  business rules pass.
- **Compatibility layer** (`src/cam_adapter.py`) bridges `forecast_base` → legacy ML
  sales and `inventory_analytics` → legacy risk snapshots without rebuilding either
  engine.

---
**End of Phase 4.** Next step (after approval): **Phase 5 — Exploratory Data Analysis**.